# LLM Results Analysis

In [89]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

In [90]:
main_pattern = re.compile(r'^(?P<dataset>.+)_(?P<method>baseline|noPrototype|noPrototypes|rulebased|jumbled)_(?P<k>\d+)_(?P<num_rules>\d+)_llm_results\.jsonl$')

candidate_bases = {
    Path('.'),
    Path('llm_results'),
    Path('../llm_results'),
    Path('results/llm_results'),
    Path('../results/llm_results'),
    Path.home() / 'Documents' / 'GitHub' / 'TSC_LLM_rules' / 'results' / 'llm_results',
}

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate_bases.add(parent / 'results' / 'llm_results')
    candidate_bases.add(parent / 'llm_results')

def count_main_named_files(candidate):
    if not candidate.exists() or not candidate.is_dir():
        return -1
    count = 0
    for path in candidate.rglob('*.jsonl'):
        if path.parent.name == 'old':
            continue
        if main_pattern.match(path.name):
            count += 1
    return count

scored_bases = []
for candidate in sorted(candidate_bases):
    score = count_main_named_files(candidate)
    if score >= 0:
        scored_bases.append((score, 'Trash' not in str(candidate), candidate))

if not scored_bases:
    raise ValueError('Could not find any candidate llm_results directory.')

score, _, base = max(scored_bases)
if score == 0:
    raise ValueError('Found candidate directories, but none contain files matching dataset_method_k_numrules_llm_results.jsonl.')

rows = []

def build_file_info(path, row):
    row_method = row.get('mode', '')
    if row_method == 'noPrototypes':
        row_method = 'noPrototype'

    canonical_name = f"{row['dataset']}_{row_method}_{int(row['k'])}_{int(row['num_rules'])}_llm_results.jsonl"
    match = main_pattern.match(path.name)

    if match:
        file_dataset = match.group('dataset')
        file_method = match.group('method')
        if file_method == 'noPrototypes':
            file_method = 'noPrototype'
        file_k = int(match.group('k'))
        file_num_rules = int(match.group('num_rules'))
    else:
        file_dataset = row['dataset']
        file_method = row_method
        file_k = int(row['k'])
        file_num_rules = int(row['num_rules'])

    is_main_file = path.name == canonical_name
    variant = 'main' if is_main_file else path.stem.replace('_llm_results', '')

    return {
        'dataset_from_filename': file_dataset,
        'mode_from_filename': file_method,
        'k_from_filename': file_k,
        'num_rules_from_filename': file_num_rules,
        'variant': variant,
        'is_main_file': is_main_file,
    }

for path in sorted(base.rglob('*.jsonl')):
    if path.name == 'old_llm_rules_results.jsonl' or path.parent.name == 'old':
        continue

    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        info = build_file_info(path, row)
        row['source_file'] = path.name
        row['source_relpath'] = path.relative_to(base).as_posix()
        row['source_bucket'] = path.parent.name
        row.update(info)
        if row['is_main_file']:
            row['dataset'] = info['dataset_from_filename']
            row['mode'] = info['mode_from_filename']
            row['k'] = info['k_from_filename']
            row['num_rules'] = info['num_rules_from_filename']
        rows.append(row)

df = pd.DataFrame(rows)
df = df.sort_values(['dataset', 'mode', 'k', 'num_rules', 'source_relpath']).reset_index(drop=True)

analysis_df = df[df['is_main_file']].copy().reset_index(drop=True)

print(f'Using results base: {base.resolve()}')
print(f'Matching main-pattern files in base: {score}')
print(f'Total runs loaded: {len(df)}')
print(f'Runs used in analysis: {len(analysis_df)}')
print(f'Main files found: {df[["source_relpath", "is_main_file"]].drop_duplicates()["is_main_file"].sum()}')
print('\nFiles found:')
display(
    df[['source_relpath', 'dataset_from_filename', 'mode_from_filename', 'k_from_filename', 'num_rules_from_filename', 'variant', 'is_main_file']]
    .drop_duplicates()
    .sort_values('source_relpath')
    .reset_index(drop=True)
)

display(analysis_df.head())

if analysis_df.empty:
    raise ValueError('analysis_df is empty. Check the base directory above and the listed files/flags.')

Using results base: /home/felix/Documents/GitHub/TSC_LLM_rules/results/llm_results
Matching main-pattern files in base: 66
Total runs loaded: 669
Runs used in analysis: 649
Main files found: 64

Files found:


,source_relpath,dataset_from_filename,mode_from_filename,k_from_filename,num_rules_from_filename,variant,is_main_file
0,Chinatown_baseline_3_0_llm_results.jsonl,Chinatown,baseline,3,0,main,True
1,Chinatown_baseline_4_0_llm_results.jsonl,Chinatown,baseline,4,0,main,True
2,Chinatown_noPrototype_3_3_llm_results.jsonl,Chinatown,noPrototype,3,3,main,True
3,Chinatown_rulebased_1_1_llm_results.jsonl,Chinatown,rulebased,1,1,main,True
4,Chinatown_rulebased_1_2_llm_results.jsonl,Chinatown,rulebased,1,2,main,True
...,...,...,...,...,...,...,...
61,UMD_rulebased_4_2_llm_results.jsonl,UMD,rulebased,4,2,main,True
62,UMD_rulebased_4_3_llm_results.jsonl,UMD,rulebased,4,3,main,True
63,UMD_rulebased_4_4_llm_results.jsonl,UMD,rulebased,4,4,main,True
64,euclidean_SonyAIBORobotSurface1_rulebased_3_3_...,euclidean_SonyAIBORobotSurface1,rulebased,3,3,euclidean_SonyAIBORobotSurface1_rulebased_3_3,False


,dataset,mode,classifier,llm,k,num_rules,accuracy,extracted_rules,instance,source_file,source_relpath,source_bucket,dataset_from_filename,mode_from_filename,k_from_filename,num_rules_from_filename,variant,is_main_file
0,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.42,{},"[{'instance_id': 1, 'ts_idx': 36, 'true_label'...",Chinatown_baseline_3_0_llm_results.jsonl,Chinatown_baseline_3_0_llm_results.jsonl,llm_results,Chinatown,baseline,3,0,main,True
1,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.71,{},"[{'instance_id': 1, 'ts_idx': 112, 'true_label...",Chinatown_baseline_3_0_llm_results.jsonl,Chinatown_baseline_3_0_llm_results.jsonl,llm_results,Chinatown,baseline,3,0,main,True
2,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.79,{},"[{'instance_id': 1, 'ts_idx': 92, 'true_label'...",Chinatown_baseline_3_0_llm_results.jsonl,Chinatown_baseline_3_0_llm_results.jsonl,llm_results,Chinatown,baseline,3,0,main,True
3,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.35,{},"[{'instance_id': 1, 'ts_idx': 207, 'true_label...",Chinatown_baseline_3_0_llm_results.jsonl,Chinatown_baseline_3_0_llm_results.jsonl,llm_results,Chinatown,baseline,3,0,main,True
4,Chinatown,baseline,miniRocket,gpt-5.1,3,0,0.69,{},"[{'instance_id': 1, 'ts_idx': 151, 'true_label...",Chinatown_baseline_3_0_llm_results.jsonl,Chinatown_baseline_3_0_llm_results.jsonl,llm_results,Chinatown,baseline,3,0,main,True


In [91]:
run_counts = (
    analysis_df.groupby(['dataset', 'mode'])
      .size()
      .reset_index(name='n_runs')
      .sort_values(['dataset', 'mode'])
)

display(run_counts)

,dataset,mode,n_runs
0,Chinatown,baseline,20
1,Chinatown,noPrototype,10
2,Chinatown,rulebased,130
3,ECG200,baseline,20
4,ECG200,noPrototype,10
5,ECG200,rulebased,130
6,SonyAIBORobotSurface1,baseline,20
7,SonyAIBORobotSurface1,jumbled,10
8,SonyAIBORobotSurface1,noPrototype,19
9,SonyAIBORobotSurface1,rulebased,130


## Mean Accuracy Per Configuration

In [92]:
summary = (
    analysis_df.groupby(['dataset', 'mode', 'k', 'num_rules'])['accuracy']
      .agg(['mean', 'std', 'count', 'min', 'max'])
      .reset_index()
      .sort_values(['dataset', 'mode', 'k', 'num_rules'])
)

summary['mean'] = summary['mean'].round(3)
summary['std'] = summary['std'].round(3)
summary['min'] = summary['min'].round(3)
summary['max'] = summary['max'].round(3)

display(summary)

,dataset,mode,k,num_rules,mean,std,count,min,max
0,Chinatown,baseline,3,0,0.637,0.149,10,0.35,0.79
1,Chinatown,baseline,4,0,0.588,0.143,10,0.33,0.76
2,Chinatown,noPrototype,3,3,0.847,0.073,10,0.69,0.93
3,Chinatown,rulebased,1,1,0.814,0.173,10,0.38,0.95
4,Chinatown,rulebased,1,2,0.743,0.203,10,0.36,0.93
...,...,...,...,...,...,...,...,...,...
59,UMD,rulebased,3,3,0.756,0.041,10,0.69,0.83
60,UMD,rulebased,3,4,0.683,0.106,10,0.52,0.84
61,UMD,rulebased,4,2,0.717,0.082,10,0.60,0.83
62,UMD,rulebased,4,3,0.756,0.095,10,0.55,0.86


In [93]:
for dataset in sorted(summary['dataset'].unique()):
    print(dataset)
    display(summary[summary['dataset'] == dataset].reset_index(drop=True))

Chinatown


,dataset,mode,k,num_rules,mean,std,count,min,max
0,Chinatown,baseline,3,0,0.637,0.149,10,0.35,0.79
1,Chinatown,baseline,4,0,0.588,0.143,10,0.33,0.76
2,Chinatown,noPrototype,3,3,0.847,0.073,10,0.69,0.93
3,Chinatown,rulebased,1,1,0.814,0.173,10,0.38,0.95
4,Chinatown,rulebased,1,2,0.743,0.203,10,0.36,0.93
5,Chinatown,rulebased,1,3,0.820,0.068,10,0.67,0.91
6,Chinatown,rulebased,2,1,0.841,0.119,10,0.53,0.95
7,Chinatown,rulebased,2,2,0.885,0.064,10,0.79,0.97
8,Chinatown,rulebased,2,3,0.890,0.047,10,0.81,0.95
9,Chinatown,rulebased,3,2,0.890,0.039,10,0.84,0.94


ECG200


,dataset,mode,k,num_rules,mean,std,count,min,max
0,ECG200,baseline,3,0,0.541,0.061,10,0.44,0.65
1,ECG200,baseline,4,0,0.511,0.095,10,0.34,0.69
2,ECG200,noPrototype,3,3,0.548,0.120,10,0.32,0.69
3,ECG200,rulebased,1,1,0.612,0.111,20,0.44,0.76
4,ECG200,rulebased,1,2,0.602,0.071,10,0.49,0.74
5,ECG200,rulebased,1,3,0.592,0.101,10,0.44,0.80
6,ECG200,rulebased,2,1,0.429,0.158,10,0.28,0.77
7,ECG200,rulebased,2,2,0.457,0.142,10,0.31,0.82
8,ECG200,rulebased,2,3,0.461,0.138,10,0.31,0.72
9,ECG200,rulebased,3,2,0.625,0.072,10,0.46,0.70


SonyAIBORobotSurface1


,dataset,mode,k,num_rules,mean,std,count,min,max
0,SonyAIBORobotSurface1,baseline,3,0,0.535,0.055,10,0.45,0.61
1,SonyAIBORobotSurface1,baseline,4,0,0.508,0.041,10,0.43,0.56
2,SonyAIBORobotSurface1,jumbled,2,3,0.429,0.141,10,0.23,0.66
3,SonyAIBORobotSurface1,noPrototype,3,3,0.651,0.122,10,0.41,0.78
4,SonyAIBORobotSurface1,noPrototype,4,3,0.481,0.105,9,0.34,0.64
5,SonyAIBORobotSurface1,rulebased,1,1,0.566,0.097,10,0.39,0.69
6,SonyAIBORobotSurface1,rulebased,1,2,0.558,0.101,10,0.44,0.70
7,SonyAIBORobotSurface1,rulebased,1,3,0.653,0.091,10,0.49,0.76
8,SonyAIBORobotSurface1,rulebased,2,1,0.514,0.120,10,0.37,0.78
9,SonyAIBORobotSurface1,rulebased,2,2,0.557,0.070,10,0.44,0.66


UMD


,dataset,mode,k,num_rules,mean,std,count,min,max
0,UMD,baseline,3,0,0.405,0.053,10,0.34,0.49
1,UMD,baseline,4,0,0.397,0.053,10,0.31,0.48
2,UMD,noPrototype,3,3,0.728,0.061,10,0.61,0.80
3,UMD,rulebased,1,1,0.756,0.104,10,0.60,0.87
4,UMD,rulebased,1,2,0.778,0.062,10,0.67,0.85
5,UMD,rulebased,1,3,0.739,0.059,10,0.63,0.83
6,UMD,rulebased,2,1,0.689,0.090,10,0.60,0.92
7,UMD,rulebased,2,2,0.671,0.101,10,0.55,0.85
8,UMD,rulebased,2,3,0.664,0.100,10,0.52,0.83
9,UMD,rulebased,3,2,0.779,0.064,10,0.70,0.91


In [94]:
for dataset in sorted(summary['dataset'].unique()):
    print('=' * 80)
    print(dataset)

    dataset_summary = summary[summary['dataset'] == dataset]
    best_rule = dataset_summary[dataset_summary['mode'] == 'rulebased'].sort_values('mean', ascending=False).iloc[0]
    print(f"Best rulebased config: k={int(best_rule['k'])}, r={int(best_rule['num_rules'])}, mean={best_rule['mean']:.3f}, std={best_rule['std']:.3f}")

    baseline_rows = dataset_summary[dataset_summary['mode'] == 'baseline'].sort_values('k')
    for _, row in baseline_rows.iterrows():
        print(f"Baseline k={int(row['k'])}: mean={row['mean']:.3f}")

    no_proto_rows = dataset_summary[dataset_summary['mode'] == 'noPrototype'].sort_values(['k', 'num_rules'])
    for _, row in no_proto_rows.iterrows():
        print(f"noPrototype k={int(row['k'])}, r={int(row['num_rules'])}: mean={row['mean']:.3f}")

    if not baseline_rows.empty:
        best_baseline = baseline_rows.sort_values('mean', ascending=False).iloc[0]
        print(f"Delta best_rulebased - best_baseline: {(best_rule['mean'] - best_baseline['mean']):+.3f}")

    if not no_proto_rows.empty:
        best_no_proto = no_proto_rows.sort_values('mean', ascending=False).iloc[0]
        print(f"Delta best_rulebased - best_noPrototype: {(best_rule['mean'] - best_no_proto['mean']):+.3f}")


Chinatown
Best rulebased config: k=4, r=4, mean=0.901, std=0.049
Baseline k=3: mean=0.637
Baseline k=4: mean=0.588
noPrototype k=3, r=3: mean=0.847
Delta best_rulebased - best_baseline: +0.264
Delta best_rulebased - best_noPrototype: +0.054
ECG200
Best rulebased config: k=3, r=2, mean=0.625, std=0.072
Baseline k=3: mean=0.541
Baseline k=4: mean=0.511
noPrototype k=3, r=3: mean=0.548
Delta best_rulebased - best_baseline: +0.084
Delta best_rulebased - best_noPrototype: +0.077
SonyAIBORobotSurface1
Best rulebased config: k=3, r=5, mean=0.721, std=0.078
Baseline k=3: mean=0.535
Baseline k=4: mean=0.508
noPrototype k=3, r=3: mean=0.651
noPrototype k=4, r=3: mean=0.481
Delta best_rulebased - best_baseline: +0.186
Delta best_rulebased - best_noPrototype: +0.070
UMD
Best rulebased config: k=3, r=2, mean=0.779, std=0.064
Baseline k=3: mean=0.405
Baseline k=4: mean=0.397
noPrototype k=3, r=3: mean=0.728
Delta best_rulebased - best_baseline: +0.374
Delta best_rulebased - best_noPrototype: +0.051


## LaTeX Tables By k

In [95]:
dataset_order = ['Chinatown', 'UMD', 'ECG200', 'SonyAIBORobotSurface1']
dataset_labels = {
    'Chinatown': 'Chinatown',
    'UMD': 'UMD',
    'ECG200': 'ECG200',
    'SonyAIBORobotSurface1': 'Sony..1',
}

table_summary = summary[summary['mode'].isin(['baseline', 'noPrototype', 'rulebased'])].copy()
table_summary['config'] = table_summary.apply(
    lambda row: 'baseline'
    if row['mode'] == 'baseline'
    else (f'random (r={int(row["num_rules"])})' if row['mode'] == 'noPrototype' else f'r={int(row["num_rules"])}'),
    axis=1
)

all_k_values = sorted(table_summary['k'].unique())

for k_value in all_k_values:
    subset = table_summary[table_summary['k'] == k_value].copy()

    row_order = ['baseline']
    row_order += sorted(subset.loc[subset['mode'] == 'noPrototype', 'config'].unique(), key=lambda x: int(x.split('=')[-1].rstrip(')')))
    row_order += sorted(subset.loc[subset['mode'] == 'rulebased', 'config'].unique(), key=lambda x: int(x.split('=')[-1]))

    if not row_order:
        continue

    mean_df = (
        subset.pivot_table(index='config', columns='dataset', values='mean', aggfunc='first')
        .reindex(index=row_order)
        .reindex(columns=dataset_order)
        .rename(columns=dataset_labels)
    )
    std_df = (
        subset.pivot_table(index='config', columns='dataset', values='std', aggfunc='first')
        .reindex(index=row_order)
        .reindex(columns=dataset_order)
        .rename(columns=dataset_labels)
    )
    table_df = mean_df.copy().astype(object)
    for row_name in table_df.index:
        for col_name in table_df.columns:
            mean_val = mean_df.loc[row_name, col_name]
            std_val = std_df.loc[row_name, col_name]
            if pd.notna(mean_val):
                table_df.loc[row_name, col_name] = f'{mean_val:.0%} ± {std_val:.0%}' if pd.notna(std_val) else f'{mean_val:.0%}'
            else:
                table_df.loc[row_name, col_name] = '-'
    table_df.index.name = 'Config'

    print(f'Mean accuracy ± std for k={int(k_value)}')
    display(table_df)
    print()
    latex_table = table_df.to_latex(
        escape=False,
        column_format='l' + 'c' * len(table_df.columns),
        caption=f'Mean accuracy ± std for k={int(k_value)}',
        label=f'tab:mean_accuracy_k_{int(k_value)}'
    )
    print(latex_table)
    print('\n' + '=' * 100 + '\n')

Mean accuracy ± std for k=1


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,-,-,-,-
r=1,81% ± 17%,76% ± 10%,61% ± 11%,57% ± 10%
r=2,74% ± 20%,78% ± 6%,60% ± 7%,56% ± 10%
r=3,82% ± 7%,74% ± 6%,59% ± 10%,65% ± 9%



\begin{table}
\caption{Mean accuracy ± std for k=1}
\label{tab:mean_accuracy_k_1}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & - & - & - & - \\
r=1 & 81% ± 17% & 76% ± 10% & 61% ± 11% & 57% ± 10% \\
r=2 & 74% ± 20% & 78% ± 6% & 60% ± 7% & 56% ± 10% \\
r=3 & 82% ± 7% & 74% ± 6% & 59% ± 10% & 65% ± 9% \\
\bottomrule
\end{tabular}
\end{table}



Mean accuracy ± std for k=2


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,-,-,-,-
r=1,84% ± 12%,69% ± 9%,43% ± 16%,51% ± 12%
r=2,88% ± 6%,67% ± 10%,46% ± 14%,56% ± 7%
r=3,89% ± 5%,66% ± 10%,46% ± 14%,61% ± 8%



\begin{table}
\caption{Mean accuracy ± std for k=2}
\label{tab:mean_accuracy_k_2}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & - & - & - & - \\
r=1 & 84% ± 12% & 69% ± 9% & 43% ± 16% & 51% ± 12% \\
r=2 & 88% ± 6% & 67% ± 10% & 46% ± 14% & 56% ± 7% \\
r=3 & 89% ± 5% & 66% ± 10% & 46% ± 14% & 61% ± 8% \\
\bottomrule
\end{tabular}
\end{table}



Mean accuracy ± std for k=3


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,64% ± 15%,40% ± 5%,54% ± 6%,54% ± 6%
random (r=3),85% ± 7%,73% ± 6%,55% ± 12%,65% ± 12%
r=2,89% ± 4%,78% ± 6%,62% ± 7%,62% ± 11%
r=3,84% ± 5%,76% ± 4%,55% ± 14%,65% ± 12%
r=4,87% ± 6%,68% ± 11%,57% ± 8%,66% ± 11%
r=5,80% ± 6%,-,-,72% ± 8%



\begin{table}
\caption{Mean accuracy ± std for k=3}
\label{tab:mean_accuracy_k_3}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & 64% ± 15% & 40% ± 5% & 54% ± 6% & 54% ± 6% \\
random (r=3) & 85% ± 7% & 73% ± 6% & 55% ± 12% & 65% ± 12% \\
r=2 & 89% ± 4% & 78% ± 6% & 62% ± 7% & 62% ± 11% \\
r=3 & 84% ± 5% & 76% ± 4% & 55% ± 14% & 65% ± 12% \\
r=4 & 87% ± 6% & 68% ± 11% & 57% ± 8% & 66% ± 11% \\
r=5 & 80% ± 6% & - & - & 72% ± 8% \\
\bottomrule
\end{tabular}
\end{table}



Mean accuracy ± std for k=4


dataset,Chinatown,UMD,ECG200,Sony..1
Config,,,,
baseline,59% ± 14%,40% ± 5%,51% ± 10%,51% ± 4%
random (r=3),-,-,-,48% ± 10%
r=2,86% ± 6%,72% ± 8%,56% ± 7%,43% ± 8%
r=3,88% ± 5%,76% ± 10%,51% ± 8%,39% ± 12%
r=4,90% ± 5%,76% ± 5%,60% ± 13%,36% ± 4%



\begin{table}
\caption{Mean accuracy ± std for k=4}
\label{tab:mean_accuracy_k_4}
\begin{tabular}{lcccc}
\toprule
dataset & Chinatown & UMD & ECG200 & Sony..1 \\
Config &  &  &  &  \\
\midrule
baseline & 59% ± 14% & 40% ± 5% & 51% ± 10% & 51% ± 4% \\
random (r=3) & - & - & - & 48% ± 10% \\
r=2 & 86% ± 6% & 72% ± 8% & 56% ± 7% & 43% ± 8% \\
r=3 & 88% ± 5% & 76% ± 10% & 51% ± 8% & 39% ± 12% \\
r=4 & 90% ± 5% & 76% ± 5% & 60% ± 13% & 36% ± 4% \\
\bottomrule
\end{tabular}
\end{table}



